# Eddy Identification With Synthetic Gaussian SSH Fields (OCNG 489/689, Spring 2026)


This notebook builds a **fully reproducible machine-learning demo** for identifying ocean eddies from **synthetic sea-surface-height (SSH)** fields.

The notebook is intentionally designed for the class OCNG 489/689 Study the Oceans from Space taught at Texas A&M University by Dr. Jinbo Wang.

Here we focus on:

- start from **physical intuition**
- generate **controlled synthetic data**
- train two ML formulations:
  1. **pixel-wise segmentation**
  2. **object-level regression** for eddy center and radius
- test how performance changes with **eddy size**, **noise**, and **sampling sparsity**

## Use synthetic Gaussian eddies

Real ocean eddies are not perfect Gaussians, and real SSH fields contain fronts, filaments, internal tides, anisotropy, advection, and multiscale background variability. But Gaussian eddies are a useful **idealized control problem** because they isolate a few interpretable parameters:

- **amplitude** $A$: eddy strength
- **radius** $\sigma$: eddy size
- **center** $(x_0,y_0)$: eddy location
- **superposition**: multiple eddies in one scene

This lets us ask a clean question:

> *If an eddy is present in SSH, what scales and sampling patterns allow a model to identify it reliably?*

## Scientific context

Mesoscale eddies are ubiquitous, coherent rotating features that are clearly visible in satellite SSH and typically have horizontal scales of order $10^1$–$10^2$ km. They are dynamically important because they transport heat, carbon, nutrients, and momentum and account for a large fraction of upper-ocean variability ([Chelton et al., 2011](https://doi.org/10.1016/j.pocean.2011.01.002)). Conventional nadir altimetry revolutionized SSH mapping, but resolving the shorter wavelengths relevant to mesoscale and submesoscale dynamics remains a central observing challenge ([Fu & Ferrari, 2008](https://doi.org/10.1029/2008EO480003)).

NASA/JPL's **SWOT** mission was designed to extend ocean SSH mapping toward much finer two-dimensional scales. The mission objective is to characterize mesoscale and submesoscale circulation at roughly **15–200 km** scales, using a **wide swath** plus a nadir measurement, rather than only a one-dimensional nadir track ([SWOT JPL Oceanography](https://swot.jpl.nasa.gov/science/oceanography/); [PO.DAAC SWOT](https://podaac.jpl.nasa.gov/SWOT)).

## Roadmap

1. Conceptual introduction  
2. Gaussian eddy generator  
3. Dataset construction  
4. Visualization of training data  
5. ML model implementation  
6. Training  
7. Evaluation  
8. Sensitivity experiments  
9. Oceanographic interpretation  
10. Exercises

In [ ]:
import os
import math
import random
from dataclasses import dataclass

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("Torch threads:", torch.get_num_threads())

FAST_DEMO = True
if FAST_DEMO:
    SEG_TRAIN_SAMPLES = 384
    SEG_VAL_SAMPLES = 96
    SEG_EPOCHS = 8

    REG_TRAIN_SAMPLES = 512
    REG_VAL_SAMPLES = 128
    REG_EPOCHS = 10

    SWEEP_SAMPLES = 80
else:
    SEG_TRAIN_SAMPLES = 1200
    SEG_VAL_SAMPLES = 240
    SEG_EPOCHS = 14

    REG_TRAIN_SAMPLES = 1500
    REG_VAL_SAMPLES = 300
    REG_EPOCHS = 18

    SWEEP_SAMPLES = 160

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["image.origin"] = "lower"

## 1. Conceptual introduction

We represent an SSH anomaly as an idealized Gaussian eddy:

$$
\eta(x,y) = A \exp\!\left[-\frac{(x-x_0)^2 + (y-y_0)^2}{2\sigma^2}\right].
$$

Interpretation:

- $A$: SSH anomaly amplitude  
- $\sigma$: eddy core size  
- $(x_0,y_0)$: eddy center  

A **smaller $\sigma$** gives a sharper, narrower eddy.  
A **larger $\sigma$** gives a broader feature.  
A multi-eddy SSH field is just a **sum** of several individual eddies plus background structure and noise:

$$
\eta_{\mathrm{total}}(x,y) = \sum_{i=1}^{N_\mathrm{eddy}} \eta_i(x,y) + \eta_{\mathrm{background}}(x,y) + \eta_{\mathrm{noise}}(x,y).
$$

In this notebook, $\sigma$ is a **proxy** for eddy scale. It is not derived from stratification or latitude, so it should be interpreted as a pedagogical surrogate for changes in Rossby-scale eddy size, not a full dynamical model.

In [ ]:
A = 0.20  # m
r = np.linspace(0, 120, 500)  # km
sigmas = [12, 24, 40]

fig, ax = plt.subplots()
for sigma in sigmas:
    eta = A * np.exp(-(r**2) / (2 * sigma**2))
    ax.plot(r, eta, label=fr"$\sigma={sigma}$ km")

ax.set_xlabel("Distance from eddy center (km)")
ax.set_ylabel("SSH anomaly (m)")
ax.set_title("Gaussian eddy profiles: narrower vs broader eddies")
ax.legend()
plt.show()

## 2. Gaussian eddy generator

We now build a synthetic generator with the following ingredients:

- one to several Gaussian eddies
- variable amplitude, size, and center
- optional broad background gradient
- white noise
- correlated noise
- stylized satellite sampling masks:
  - **full** observation
  - **swath-like** observation with a central gap (SWOT-inspired)
  - **nadir-like** observation as a thin one-dimensional track

### Important scientific note

The sampling masks here are **stylized local proxies** rather than orbital simulators. Their purpose is to expose the ML model to the central observing idea:

- a **2D swath** contains much more spatial information than
- a **1D nadir track**

That distinction is central to eddy observability from satellite SSH.

In [ ]:
@dataclass
class EddyConfig:
    nx: int = 48
    ny: int = 48
    dx_km: float = 5.0
    sigma_range_km: tuple = (12.0, 50.0)
    amplitude_range_m: tuple = (0.05, 0.30)
    n_eddies_range: tuple = (1, 3)
    mask_level: float = 0.20
    background_slope_range: tuple = (-2e-4, 2e-4)
    white_noise_range_m: tuple = (0.0, 0.04)
    corr_noise_range_m: tuple = (0.0, 0.03)
    corr_length_range_km: tuple = (8.0, 20.0)
    norm_scale_m: float = 0.25
    train_sampling_modes: tuple = ("full", "swath", "nadir")
    train_sampling_probs: tuple = (0.50, 0.35, 0.15)


CFG = EddyConfig()

def make_grid(cfg=CFG):
    x = (np.arange(cfg.nx) - (cfg.nx - 1) / 2) * cfg.dx_km
    y = (np.arange(cfg.ny) - (cfg.ny - 1) / 2) * cfg.dx_km
    X, Y = np.meshgrid(x, y)
    return X, Y

X, Y = make_grid(CFG)
DOMAIN_X_KM = CFG.nx * CFG.dx_km
DOMAIN_Y_KM = CFG.ny * CFG.dx_km
SIGMA_MIN_KM, SIGMA_MAX_KM = CFG.sigma_range_km

def gaussian_correlated_noise(rng, shape, dx_km, corr_length_km, std):
    # Smooth correlated noise via a Gaussian low-pass filter in Fourier space.
    if std <= 0:
        return np.zeros(shape, dtype=np.float32)

    z = rng.normal(size=shape)
    ky = np.fft.fftfreq(shape[0], d=dx_km)
    kx = np.fft.fftfreq(shape[1], d=dx_km)
    KX, KY = np.meshgrid(kx, ky)

    filt = np.exp(-2 * (np.pi**2) * (corr_length_km**2) * (KX**2 + KY**2))
    out = np.fft.ifft2(np.fft.fft2(z) * filt).real
    out = out / (out.std() + 1e-6) * std
    return out.astype(np.float32)

def make_sampling_mask(rng, cfg=CFG, mode="full"):
    # Stylized observing masks used for demonstration.
    mask = np.ones((cfg.ny, cfg.nx), dtype=np.float32)

    if mode == "full":
        return mask

    orientation = rng.integers(0, 2)

    if mode == "swath":
        gap = int(max(4, round(0.12 * cfg.nx)))
        if orientation == 0:
            c = cfg.nx // 2 + rng.integers(-2, 3)
            mask[:, max(0, c - gap // 2):min(cfg.nx, c + gap // 2)] = 0.0
        else:
            c = cfg.ny // 2 + rng.integers(-2, 3)
            mask[max(0, c - gap // 2):min(cfg.ny, c + gap // 2), :] = 0.0
        return mask

    if mode == "nadir":
        width = 1
        mask[:] = 0.0
        if orientation == 0:
            c = cfg.nx // 2 + rng.integers(-4, 5)
            mask[:, max(0, c - width):min(cfg.nx, c + width + 1)] = 1.0
        else:
            c = cfg.ny // 2 + rng.integers(-4, 5)
            mask[max(0, c - width):min(cfg.ny, c + width + 1), :] = 1.0
        return mask

    if mode == "decimate":
        stride = 4
        mask[:] = 0.0
        if orientation == 0:
            mask[:, ::stride] = 1.0
        else:
            mask[::stride, :] = 1.0
        return mask

    raise ValueError(f"Unknown sampling mode: {mode}")

def sample_scene(
    rng,
    cfg=CFG,
    X=X,
    Y=Y,
    n_eddies=None,
    single_only=False,
    sigma_override=None,
    white_noise_std=None,
    corr_noise_std=None,
    sampling_mode="full",
    allow_background=True,
):
    # Generate one synthetic SSH scene and associated labels/metadata.
    if n_eddies is None:
        n_eddies = 1 if single_only else rng.integers(cfg.n_eddies_range[0], cfg.n_eddies_range[1] + 1)

    field = np.zeros_like(X, dtype=np.float32)
    y_mask = np.zeros_like(X, dtype=np.float32)
    eddies = []

    Lx = cfg.nx * cfg.dx_km
    Ly = cfg.ny * cfg.dx_km
    mask_radius_factor = math.sqrt(-2.0 * math.log(cfg.mask_level))

    for _ in range(n_eddies):
        sigma = sigma_override if sigma_override is not None else rng.uniform(*cfg.sigma_range_km)
        amp = rng.uniform(*cfg.amplitude_range_m) * (1.0 if rng.random() > 0.5 else -1.0)

        margin_x = min(0.35 * Lx, 2.5 * sigma)
        margin_y = min(0.35 * Ly, 2.5 * sigma)
        x0 = rng.uniform(-Lx / 2 + margin_x, Lx / 2 - margin_x)
        y0 = rng.uniform(-Ly / 2 + margin_y, Ly / 2 - margin_y)

        r2 = (X - x0) ** 2 + (Y - y0) ** 2
        contribution = amp * np.exp(-r2 / (2 * sigma**2))
        field += contribution.astype(np.float32)

        y_mask = np.maximum(
            y_mask,
            (r2 <= (mask_radius_factor * sigma) ** 2).astype(np.float32),
        )

        eddies.append(
            {
                "x0_km": float(x0),
                "y0_km": float(y0),
                "sigma_km": float(sigma),
                "amplitude_m": float(amp),
            }
        )

    field_clean = field.copy()

    if allow_background:
        ax = rng.uniform(*cfg.background_slope_range)
        ay = rng.uniform(*cfg.background_slope_range)
        c0 = rng.uniform(-0.03, 0.03)
        field = field + (ax * X + ay * Y + c0).astype(np.float32)

    if white_noise_std is None:
        white_noise_std = rng.uniform(*cfg.white_noise_range_m)
    if corr_noise_std is None:
        corr_noise_std = rng.uniform(*cfg.corr_noise_range_m)

    field = field + rng.normal(scale=white_noise_std, size=field.shape).astype(np.float32)
    field = field + gaussian_correlated_noise(
        rng,
        field.shape,
        cfg.dx_km,
        rng.uniform(*cfg.corr_length_range_km),
        corr_noise_std,
    )

    if sampling_mode == "mixed":
        sampling_mode = rng.choice(cfg.train_sampling_modes, p=cfg.train_sampling_probs)

    obs_mask = make_sampling_mask(rng, cfg, sampling_mode)
    field_obs = field * obs_mask

    x = np.stack([field_obs / cfg.norm_scale_m, obs_mask], axis=0).astype(np.float32)
    y = y_mask[None].astype(np.float32)

    meta = {
        "eddies": eddies,
        "sampling_mode": sampling_mode,
        "mean_sigma_km": float(np.mean([e["sigma_km"] for e in eddies])),
        "white_noise_std": float(white_noise_std),
        "corr_noise_std": float(corr_noise_std),
    }
    return x, y, meta, field_clean, field_obs

def km_to_pixel_x(x_km, cfg=CFG):
    return (x_km / cfg.dx_km) + (cfg.nx - 1) / 2

def km_to_pixel_y(y_km, cfg=CFG):
    return (y_km / cfg.dx_km) + (cfg.ny - 1) / 2

## 3. Dataset construction

We will use **two synthetic datasets**:

### (A) Segmentation dataset
- scenes with **1–3 eddies**
- target is a **binary eddy mask**
- used to train a small **U-Net-like** model

### (B) Regression dataset
- scenes with **exactly 1 eddy**
- target is:
  - eddy center $(x_0,y_0)$
  - eddy size $\sigma$
- used to train a simple **CNN regressor**

### Why restrict regression to one eddy?

Direct regression of **one** center and **one** radius is only well-posed when there is one dominant target. In multi-eddy scenes, “the eddy center” is ambiguous unless the architecture is extended to full detection or set prediction. That ambiguity is actually useful pedagogically and we will return to it later.

In [ ]:
XX_NORM = ((X + DOMAIN_X_KM / 2) / DOMAIN_X_KM).astype(np.float32)
YY_NORM = ((Y + DOMAIN_Y_KM / 2) / DOMAIN_Y_KM).astype(np.float32)
COORD_CHANNELS = np.stack([XX_NORM, YY_NORM], axis=0)

def build_segmentation_tensors(n_samples, seed=0, sampling_mode="mixed", single_only=False):
    xs, ys, metas = [], [], []
    for i in range(n_samples):
        rng = np.random.default_rng(seed + i)
        x, y, meta, _, _ = sample_scene(
            rng,
            cfg=CFG,
            X=X,
            Y=Y,
            sampling_mode=sampling_mode,
            single_only=single_only,
        )
        xs.append(x)
        ys.append(y)
        metas.append(meta)
    return torch.tensor(np.stack(xs)), torch.tensor(np.stack(ys)), metas

def build_regression_tensors(n_samples, seed=0, sampling_mode="full"):
    xs, ys, metas = [], [], []
    for i in range(n_samples):
        rng = np.random.default_rng(seed + i)
        x, _, meta, _, _ = sample_scene(
            rng,
            cfg=CFG,
            X=X,
            Y=Y,
            sampling_mode=sampling_mode,
            single_only=True,
        )
        x = np.concatenate([x, COORD_CHANNELS], axis=0)

        eddy = meta["eddies"][0]
        tx = (eddy["x0_km"] + DOMAIN_X_KM / 2) / DOMAIN_X_KM
        ty = (eddy["y0_km"] + DOMAIN_Y_KM / 2) / DOMAIN_Y_KM
        ts = (eddy["sigma_km"] - SIGMA_MIN_KM) / (SIGMA_MAX_KM - SIGMA_MIN_KM)

        xs.append(x)
        ys.append([tx, ty, ts])
        metas.append(meta)

    return torch.tensor(np.stack(xs)), torch.tensor(np.array(ys, dtype=np.float32)), metas

## 4. Visualization of training data

Before training any model, inspect the synthetic scenes.

We want to see:

- the SSH anomaly field
- the binary eddy label
- the observation mask

This is important: **if the synthetic generator is not interpretable, the ML results will not be interpretable either.**

In [ ]:
def plot_scene_examples():
    fig, axes = plt.subplots(3, 3, figsize=(8, 8))
    modes = ["full", "swath", "nadir"]

    for row, mode in enumerate(modes):
        rng = np.random.default_rng(100 + row)
        x, y, meta, field_clean, field_obs = sample_scene(
            rng,
            cfg=CFG,
            X=X,
            Y=Y,
            sampling_mode=mode,
            single_only=False,
        )

        ax = axes[row, 0]
        im = ax.imshow(field_obs, extent=[X.min(), X.max(), Y.min(), Y.max()])
        ax.set_title(f"Observed SSH ({mode})")
        ax.set_xlabel("x (km)")
        ax.set_ylabel("y (km)")
        plt.colorbar(im, ax=ax, fraction=0.046)

        ax = axes[row, 1]
        ax.imshow(y[0], extent=[X.min(), X.max(), Y.min(), Y.max()], vmin=0, vmax=1)
        ax.set_title("Ground-truth eddy mask")
        ax.set_xlabel("x (km)")
        ax.set_ylabel("y (km)")

        ax = axes[row, 2]
        ax.imshow(x[1], extent=[X.min(), X.max(), Y.min(), Y.max()], vmin=0, vmax=1)
        ax.set_title("Observation mask")
        ax.set_xlabel("x (km)")
        ax.set_ylabel("y (km)")

    plt.tight_layout()
    plt.show()

plot_scene_examples()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3))

for i, n_eddies in enumerate([1, 2, 3]):
    rng = np.random.default_rng(200 + i)
    x, y, meta, field_clean, field_obs = sample_scene(
        rng,
        cfg=CFG,
        X=X,
        Y=Y,
        n_eddies=n_eddies,
        sampling_mode="full",
    )
    im = axes[i].imshow(field_clean, extent=[X.min(), X.max(), Y.min(), Y.max()])
    axes[i].set_title(f"{n_eddies} synthetic eddy(ies)")
    axes[i].set_xlabel("x (km)")
    axes[i].set_ylabel("y (km)")
    plt.colorbar(im, ax=axes[i], fraction=0.046)

plt.tight_layout()
plt.show()

## 5. ML model implementation

We now implement two lightweight neural networks.

### Segmentation model
A small **U-Net-like** network:
- input channels: observed SSH, observation mask
- output: per-pixel probability of “eddy core”

### Regression model
A compact CNN:
- input channels: observed SSH, observation mask, normalized $x$, normalized $y$
- output: $(x_0, y_0, \sigma)$

### Losses
- Segmentation: **BCE + Dice loss**
- Regression: **mean-squared error (MSE)**

The segmentation problem is class-imbalanced because eddy pixels occupy only part of the domain, so we use a positive-class weighting in BCE.

In [ ]:
class TinyUNet(nn.Module):
    def __init__(self, in_channels=2, base=10):
        super().__init__()
        self.enc1 = nn.Sequential(
            nn.Conv2d(in_channels, base, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(base, base, 3, padding=1),
            nn.ReLU(),
        )
        self.pool = nn.MaxPool2d(2)
        self.enc2 = nn.Sequential(
            nn.Conv2d(base, base * 2, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(base * 2, base * 2, 3, padding=1),
            nn.ReLU(),
        )
        self.up = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.dec = nn.Sequential(
            nn.Conv2d(base * 2, base, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(base, base, 3, padding=1),
            nn.ReLU(),
        )
        self.out = nn.Conv2d(base, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        u = self.up(e2)
        d = self.dec(torch.cat([u, e1], dim=1))
        return self.out(d)


class EddyRegressor(nn.Module):
    def __init__(self, in_channels=4, base=8):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, base, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(base, base * 2, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(base * 2, base * 4, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(base * 4, base * 4, 3, padding=1),
            nn.ReLU(),
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(base * 4 * 6 * 6, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 3),
        )

    def forward(self, x):
        return torch.sigmoid(self.fc(self.conv(x)))


def dice_loss_with_logits(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    numer = 2.0 * (probs * targets).sum(dim=(1, 2, 3)) + eps
    denom = probs.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3)) + eps
    return (1.0 - numer / denom).mean()


def segmentation_metrics_from_logits(logits, targets):
    probs = torch.sigmoid(logits)
    preds = (probs > 0.5).float()

    inter = (preds * targets).sum(dim=(1, 2, 3))
    union = ((preds + targets) > 0).float().sum(dim=(1, 2, 3))
    iou = (inter + 1e-6) / (union + 1e-6)

    pixel_acc = preds.eq(targets).float().mean(dim=(1, 2, 3))
    detection = (iou > 0.5).float()

    return {
        "iou": float(iou.mean().item()),
        "accuracy": float(pixel_acc.mean().item()),
        "detection": float(detection.mean().item()),
    }


def regression_metrics(pred, target):
    center_dx_km = (pred[:, 0] - target[:, 0]) * DOMAIN_X_KM
    center_dy_km = (pred[:, 1] - target[:, 1]) * DOMAIN_Y_KM
    center_error_km = torch.sqrt(center_dx_km**2 + center_dy_km**2)

    sigma_error_km = torch.abs(pred[:, 2] - target[:, 2]) * (SIGMA_MAX_KM - SIGMA_MIN_KM)

    return {
        "center_error_km": float(center_error_km.mean().item()),
        "sigma_error_km": float(sigma_error_km.mean().item()),
    }


def evaluate_segmentation_model(model, dataloader, pos_weight=4.0):
    model.eval()
    losses, ious, accs, dets = [], [], [], []

    pw = torch.tensor([pos_weight], device=DEVICE)

    with torch.no_grad():
        for xb, yb in dataloader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            logits = model(xb)
            loss = F.binary_cross_entropy_with_logits(logits, yb, pos_weight=pw) + dice_loss_with_logits(logits, yb)
            losses.append(loss.item())

            metrics = segmentation_metrics_from_logits(logits, yb)
            ious.append(metrics["iou"])
            accs.append(metrics["accuracy"])
            dets.append(metrics["detection"])

    return {
        "loss": float(np.mean(losses)),
        "iou": float(np.mean(ious)),
        "accuracy": float(np.mean(accs)),
        "detection": float(np.mean(dets)),
    }


def evaluate_regression_model(model, dataloader):
    model.eval()
    losses, center_errors, sigma_errors = [], [], []

    with torch.no_grad():
        for xb, yb in dataloader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            pred = model(xb)
            loss = F.mse_loss(pred, yb)
            losses.append(loss.item())

            metrics = regression_metrics(pred, yb)
            center_errors.append(metrics["center_error_km"])
            sigma_errors.append(metrics["sigma_error_km"])

    return {
        "loss": float(np.mean(losses)),
        "center_error_km": float(np.mean(center_errors)),
        "sigma_error_km": float(np.mean(sigma_errors)),
    }


def train_segmentation_model(train_loader, val_loader, epochs=8, lr=1e-3, pos_weight=4.0):
    model = TinyUNet().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    pw = torch.tensor([pos_weight], device=DEVICE)

    history = {"train_loss": [], "val_loss": [], "val_iou": [], "val_accuracy": [], "val_detection": []}

    for epoch in range(epochs):
        model.train()
        train_losses = []

        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()
            logits = model(xb)
            loss = F.binary_cross_entropy_with_logits(logits, yb, pos_weight=pw) + dice_loss_with_logits(logits, yb)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        val_metrics = evaluate_segmentation_model(model, val_loader, pos_weight=pos_weight)

        history["train_loss"].append(float(np.mean(train_losses)))
        history["val_loss"].append(val_metrics["loss"])
        history["val_iou"].append(val_metrics["iou"])
        history["val_accuracy"].append(val_metrics["accuracy"])
        history["val_detection"].append(val_metrics["detection"])

        print(
            f"Epoch {epoch+1:02d}/{epochs} | "
            f"train loss={history['train_loss'][-1]:.3f} | "
            f"val IoU={val_metrics['iou']:.3f} | "
            f"val acc={val_metrics['accuracy']:.3f} | "
            f"val detect={val_metrics['detection']:.3f}"
        )

    return model, history


def train_regression_model(train_loader, val_loader, epochs=10, lr=1e-3):
    model = EddyRegressor().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {"train_loss": [], "val_loss": [], "val_center_error_km": [], "val_sigma_error_km": []}

    for epoch in range(epochs):
        model.train()
        train_losses = []

        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()
            pred = model(xb)
            loss = F.mse_loss(pred, yb)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        val_metrics = evaluate_regression_model(model, val_loader)

        history["train_loss"].append(float(np.mean(train_losses)))
        history["val_loss"].append(val_metrics["loss"])
        history["val_center_error_km"].append(val_metrics["center_error_km"])
        history["val_sigma_error_km"].append(val_metrics["sigma_error_km"])

        print(
            f"Epoch {epoch+1:02d}/{epochs} | "
            f"train loss={history['train_loss'][-1]:.4f} | "
            f"val center err={val_metrics['center_error_km']:.2f} km | "
            f"val sigma err={val_metrics['sigma_error_km']:.2f} km"
        )

    return model, history

## 6. Training

The notebook uses compact training sets so that it runs end-to-end on CPU in a reasonable time. If you want stronger performance, increase the sample counts or number of epochs.

In [ ]:
X_seg_train, Y_seg_train, META_seg_train = build_segmentation_tensors(
    SEG_TRAIN_SAMPLES, seed=100, sampling_mode="mixed", single_only=False
)
X_seg_val, Y_seg_val, META_seg_val = build_segmentation_tensors(
    SEG_VAL_SAMPLES, seed=1000, sampling_mode="mixed", single_only=False
)

seg_train_loader = DataLoader(TensorDataset(X_seg_train, Y_seg_train), batch_size=32, shuffle=True)
seg_val_loader = DataLoader(TensorDataset(X_seg_val, Y_seg_val), batch_size=32, shuffle=False)

print("Segmentation training tensor shape:", X_seg_train.shape, Y_seg_train.shape)

seg_model, seg_history = train_segmentation_model(
    seg_train_loader,
    seg_val_loader,
    epochs=SEG_EPOCHS,
    lr=1e-3,
    pos_weight=4.0,
)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8, 4))

ax[0].plot(seg_history["train_loss"], label="train")
ax[0].plot(seg_history["val_loss"], label="val")
ax[0].set_title("Segmentation loss history")
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("Loss")
ax[0].legend()

ax[1].plot(seg_history["val_iou"], label="IoU")
ax[1].plot(seg_history["val_accuracy"], label="Pixel accuracy")
ax[1].plot(seg_history["val_detection"], label="Detection accuracy")
ax[1].set_title("Segmentation validation metrics")
ax[1].set_xlabel("Epoch")
ax[1].set_ylabel("Metric")
ax[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
X_reg_train, Y_reg_train, META_reg_train = build_regression_tensors(
    REG_TRAIN_SAMPLES, seed=200, sampling_mode="full"
)
X_reg_val, Y_reg_val, META_reg_val = build_regression_tensors(
    REG_VAL_SAMPLES, seed=2000, sampling_mode="full"
)

reg_train_loader = DataLoader(TensorDataset(X_reg_train, Y_reg_train), batch_size=32, shuffle=True)
reg_val_loader = DataLoader(TensorDataset(X_reg_val, Y_reg_val), batch_size=32, shuffle=False)

print("Regression training tensor shape:", X_reg_train.shape, Y_reg_train.shape)

reg_model, reg_history = train_regression_model(
    reg_train_loader,
    reg_val_loader,
    epochs=REG_EPOCHS,
    lr=1e-3,
)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8, 4))

ax[0].plot(reg_history["train_loss"], label="train")
ax[0].plot(reg_history["val_loss"], label="val")
ax[0].set_title("Regression loss history")
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("MSE")
ax[0].legend()

ax[1].plot(reg_history["val_center_error_km"], label="Center error (km)")
ax[1].plot(reg_history["val_sigma_error_km"], label="$\sigma$ error (km)")
ax[1].set_title("Regression validation metrics")
ax[1].set_xlabel("Epoch")
ax[1].set_ylabel("Error")
ax[1].legend()

plt.tight_layout()
plt.show()

## 7. Evaluation

We now examine:

1. final validation metrics
2. example segmentation predictions
3. example regression predictions
4. some failure cases

In [ ]:
seg_val_metrics = evaluate_segmentation_model(seg_model, seg_val_loader, pos_weight=4.0)
reg_val_metrics = evaluate_regression_model(reg_model, reg_val_loader)

print("Final validation metrics")
print("-" * 40)
print("Segmentation:")
for k, v in seg_val_metrics.items():
    print(f"  {k:>12s} : {v:.3f}")

print("\nRegression:")
for k, v in reg_val_metrics.items():
    print(f"  {k:>18s} : {v:.3f}")

In [ ]:
def plot_segmentation_predictions(model, x_batch, y_batch, n_examples=4):
    model.eval()
    with torch.no_grad():
        logits = model(x_batch[:n_examples].to(DEVICE))
        probs = torch.sigmoid(logits).cpu()
        preds = (probs > 0.5).float()

    fig, axes = plt.subplots(n_examples, 4, figsize=(12, 3 * n_examples))
    if n_examples == 1:
        axes = np.expand_dims(axes, axis=0)

    for i in range(n_examples):
        axes[i, 0].imshow(x_batch[i, 0].cpu())
        axes[i, 0].set_title("Observed SSH / norm")

        axes[i, 1].imshow(x_batch[i, 1].cpu(), vmin=0, vmax=1)
        axes[i, 1].set_title("Observation mask")

        axes[i, 2].imshow(y_batch[i, 0].cpu(), vmin=0, vmax=1)
        axes[i, 2].set_title("True eddy mask")

        axes[i, 3].imshow(preds[i, 0], vmin=0, vmax=1)
        axes[i, 3].set_title("Predicted mask")

        for j in range(4):
            axes[i, j].set_xticks([])
            axes[i, j].set_yticks([])

    plt.tight_layout()
    plt.show()

plot_segmentation_predictions(seg_model, X_seg_val, Y_seg_val, n_examples=4)

In [ ]:
seg_model.eval()
with torch.no_grad():
    logits = seg_model(X_seg_val.to(DEVICE))
    probs = torch.sigmoid(logits).cpu()
    preds = (probs > 0.5).float()

inter = (preds * Y_seg_val).sum(dim=(1, 2, 3))
union = ((preds + Y_seg_val) > 0).float().sum(dim=(1, 2, 3))
sample_iou = ((inter + 1e-6) / (union + 1e-6)).numpy()

worst_idx = np.argsort(sample_iou)[:4]
print("Lowest-IoU validation samples:", worst_idx.tolist())
print("Their IoUs:", np.round(sample_iou[worst_idx], 3))

plot_segmentation_predictions(seg_model, X_seg_val[worst_idx], Y_seg_val[worst_idx], n_examples=4)

In [ ]:
def denormalize_regression_output(pred_vec):
    x_km = pred_vec[0] * DOMAIN_X_KM - DOMAIN_X_KM / 2
    y_km = pred_vec[1] * DOMAIN_Y_KM - DOMAIN_Y_KM / 2
    sigma_km = pred_vec[2] * (SIGMA_MAX_KM - SIGMA_MIN_KM) + SIGMA_MIN_KM
    return x_km, y_km, sigma_km

def plot_regression_predictions(model, x_batch, target_batch, n_examples=4):
    model.eval()
    with torch.no_grad():
        pred = model(x_batch[:n_examples].to(DEVICE)).cpu().numpy()

    fig, axes = plt.subplots(1, n_examples, figsize=(3.8 * n_examples, 4))

    if n_examples == 1:
        axes = [axes]

    for i in range(n_examples):
        ax = axes[i]
        observed = x_batch[i, 0].cpu().numpy()
        ax.imshow(observed)

        true_x_km, true_y_km, true_sigma_km = denormalize_regression_output(target_batch[i].cpu().numpy())
        pred_x_km, pred_y_km, pred_sigma_km = denormalize_regression_output(pred[i])

        true_center = (km_to_pixel_x(true_x_km), km_to_pixel_y(true_y_km))
        pred_center = (km_to_pixel_x(pred_x_km), km_to_pixel_y(pred_y_km))

        true_circle = Circle(true_center, true_sigma_km / CFG.dx_km, fill=False, linewidth=2)
        pred_circle = Circle(pred_center, pred_sigma_km / CFG.dx_km, fill=False, linewidth=2, linestyle="--")

        ax.add_patch(true_circle)
        ax.add_patch(pred_circle)
        ax.plot(*true_center, marker="x", markersize=10, mew=2)
        ax.plot(*pred_center, marker="+", markersize=12, mew=2)

        center_err = np.sqrt((pred_x_km - true_x_km) ** 2 + (pred_y_km - true_y_km) ** 2)
        sigma_err = abs(pred_sigma_km - true_sigma_km)

        ax.set_title(
            f"#{i}\ncenter err={center_err:.1f} km\nsigma err={sigma_err:.1f} km",
            fontsize=10,
        )
        ax.set_xticks([])
        ax.set_yticks([])

    fig.suptitle("Regression predictions: true (solid/x) vs predicted (dashed/+)", y=1.02)
    plt.tight_layout()
    plt.show()

plot_regression_predictions(reg_model, X_reg_val, Y_reg_val, n_examples=4)

## 8. Sensitivity experiments

Now we test the central teaching question:

> **How does eddy detectability depend on physical scale, noise, and observing pattern?**

We run three sweeps:

1. **eddy size sweep**  
2. **noise sweep**  
3. **sampling sweep**

For the size and noise experiments, we hold the problem as close as possible to “one eddy in one patch” so the trends are easy to interpret.

In [ ]:
def build_seg_batch(
    n,
    seed=0,
    sigma_override=None,
    white_noise_std=None,
    corr_noise_std=None,
    sampling_mode="full",
    single_only=True,
):
    xs, ys, metas = [], [], []
    for i in range(n):
        rng = np.random.default_rng(seed + i)
        x, y, meta, _, _ = sample_scene(
            rng,
            cfg=CFG,
            X=X,
            Y=Y,
            sigma_override=sigma_override,
            white_noise_std=white_noise_std,
            corr_noise_std=corr_noise_std,
            sampling_mode=sampling_mode,
            single_only=single_only,
        )
        xs.append(x)
        ys.append(y)
        metas.append(meta)
    return torch.tensor(np.stack(xs)), torch.tensor(np.stack(ys)), metas

def build_reg_batch(
    n,
    seed=0,
    sigma_override=None,
    white_noise_std=None,
    corr_noise_std=None,
    sampling_mode="full",
):
    xs, ys, metas = [], [], []
    for i in range(n):
        rng = np.random.default_rng(seed + i)
        x, _, meta, _, _ = sample_scene(
            rng,
            cfg=CFG,
            X=X,
            Y=Y,
            sigma_override=sigma_override,
            white_noise_std=white_noise_std,
            corr_noise_std=corr_noise_std,
            sampling_mode=sampling_mode,
            single_only=True,
        )
        x = np.concatenate([x, COORD_CHANNELS], axis=0)

        eddy = meta["eddies"][0]
        tx = (eddy["x0_km"] + DOMAIN_X_KM / 2) / DOMAIN_X_KM
        ty = (eddy["y0_km"] + DOMAIN_Y_KM / 2) / DOMAIN_Y_KM
        ts = (eddy["sigma_km"] - SIGMA_MIN_KM) / (SIGMA_MAX_KM - SIGMA_MIN_KM)

        xs.append(x)
        ys.append([tx, ty, ts])
        metas.append(meta)

    return torch.tensor(np.stack(xs)), torch.tensor(np.array(ys, dtype=np.float32)), metas

def evaluate_seg_batch(model, xb, yb):
    loader = DataLoader(TensorDataset(xb, yb), batch_size=32, shuffle=False)
    return evaluate_segmentation_model(model, loader, pos_weight=4.0)

def evaluate_reg_batch(model, xb, yb):
    loader = DataLoader(TensorDataset(xb, yb), batch_size=32, shuffle=False)
    return evaluate_regression_model(model, loader)

In [ ]:
size_values_km = [12, 20, 28, 36, 44, 50]

size_seg_iou = []
size_seg_det = []
size_reg_center = []
size_reg_sigma = []

for i, sigma_km in enumerate(size_values_km):
    xb_seg, yb_seg, _ = build_seg_batch(
        SWEEP_SAMPLES,
        seed=5000 + 100 * i,
        sigma_override=sigma_km,
        sampling_mode="full",
        single_only=True,
    )
    seg_metrics = evaluate_seg_batch(seg_model, xb_seg, yb_seg)

    xb_reg, yb_reg, _ = build_reg_batch(
        SWEEP_SAMPLES,
        seed=6000 + 100 * i,
        sigma_override=sigma_km,
        sampling_mode="full",
    )
    reg_metrics = evaluate_reg_batch(reg_model, xb_reg, yb_reg)

    size_seg_iou.append(seg_metrics["iou"])
    size_seg_det.append(seg_metrics["detection"])
    size_reg_center.append(reg_metrics["center_error_km"])
    size_reg_sigma.append(reg_metrics["sigma_error_km"])

fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].plot(size_values_km, size_seg_iou, marker="o", label="Segmentation IoU")
ax[0].plot(size_values_km, size_seg_det, marker="s", label="Detection accuracy")
ax[0].set_xlabel("True eddy sigma (km)")
ax[0].set_ylabel("Metric")
ax[0].set_title("Performance vs eddy size")
ax[0].legend()

ax[1].plot(size_values_km, size_reg_center, marker="o", label="Center error (km)")
ax[1].plot(size_values_km, size_reg_sigma, marker="s", label="Sigma error (km)")
ax[1].set_xlabel("True eddy sigma (km)")
ax[1].set_ylabel("Error (km)")
ax[1].set_title("Regression error vs eddy size")
ax[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
noise_values_m = [0.00, 0.01, 0.02, 0.04, 0.06]
noise_seg_iou = []
noise_seg_det = []
noise_reg_center = []
noise_reg_sigma = []

for i, noise_std in enumerate(noise_values_m):
    xb_seg, yb_seg, _ = build_seg_batch(
        SWEEP_SAMPLES,
        seed=7000 + 100 * i,
        sigma_override=28,
        white_noise_std=noise_std,
        corr_noise_std=0.6 * noise_std,
        sampling_mode="full",
        single_only=True,
    )
    seg_metrics = evaluate_seg_batch(seg_model, xb_seg, yb_seg)

    xb_reg, yb_reg, _ = build_reg_batch(
        SWEEP_SAMPLES,
        seed=8000 + 100 * i,
        sigma_override=28,
        white_noise_std=noise_std,
        corr_noise_std=0.6 * noise_std,
        sampling_mode="full",
    )
    reg_metrics = evaluate_reg_batch(reg_model, xb_reg, yb_reg)

    noise_seg_iou.append(seg_metrics["iou"])
    noise_seg_det.append(seg_metrics["detection"])
    noise_reg_center.append(reg_metrics["center_error_km"])
    noise_reg_sigma.append(reg_metrics["sigma_error_km"])

fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].plot(noise_values_m, noise_seg_iou, marker="o", label="Segmentation IoU")
ax[0].plot(noise_values_m, noise_seg_det, marker="s", label="Detection accuracy")
ax[0].set_xlabel("White-noise std (m)")
ax[0].set_ylabel("Metric")
ax[0].set_title("Performance vs noise level")
ax[0].legend()

ax[1].plot(noise_values_m, noise_reg_center, marker="o", label="Center error (km)")
ax[1].plot(noise_values_m, noise_reg_sigma, marker="s", label="Sigma error (km)")
ax[1].set_xlabel("White-noise std (m)")
ax[1].set_ylabel("Error (km)")
ax[1].set_title("Regression error vs noise level")
ax[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
sampling_modes = ["full", "swath", "nadir", "decimate"]

sampling_seg_iou = []
sampling_seg_det = []
sampling_reg_center = []
sampling_reg_sigma = []

for i, mode in enumerate(sampling_modes):
    xb_seg, yb_seg, _ = build_seg_batch(
        SWEEP_SAMPLES,
        seed=9000 + 100 * i,
        sigma_override=28,
        white_noise_std=0.02,
        corr_noise_std=0.012,
        sampling_mode=mode,
        single_only=True,
    )
    seg_metrics = evaluate_seg_batch(seg_model, xb_seg, yb_seg)

    xb_reg, yb_reg, _ = build_reg_batch(
        SWEEP_SAMPLES,
        seed=10000 + 100 * i,
        sigma_override=28,
        white_noise_std=0.02,
        corr_noise_std=0.012,
        sampling_mode=mode,
    )
    reg_metrics = evaluate_reg_batch(reg_model, xb_reg, yb_reg)

    sampling_seg_iou.append(seg_metrics["iou"])
    sampling_seg_det.append(seg_metrics["detection"])
    sampling_reg_center.append(reg_metrics["center_error_km"])
    sampling_reg_sigma.append(reg_metrics["sigma_error_km"])

xpos = np.arange(len(sampling_modes))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].bar(xpos - 0.18, sampling_seg_iou, width=0.36, label="Segmentation IoU")
ax[0].bar(xpos + 0.18, sampling_seg_det, width=0.36, label="Detection accuracy")
ax[0].set_xticks(xpos)
ax[0].set_xticklabels(sampling_modes)
ax[0].set_ylabel("Metric")
ax[0].set_title("Segmentation vs sampling pattern")
ax[0].legend()

ax[1].bar(xpos - 0.18, sampling_reg_center, width=0.36, label="Center error (km)")
ax[1].bar(xpos + 0.18, sampling_reg_sigma, width=0.36, label="Sigma error (km)")
ax[1].set_xticks(xpos)
ax[1].set_xticklabels(sampling_modes)
ax[1].set_ylabel("Error (km)")
ax[1].set_title("Regression vs sampling pattern")
ax[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
def build_multiedy_regression_demo_batch(n_samples=4, n_eddies=3, seed=12000):
    xs, metas = [], []
    for i in range(n_samples):
        rng = np.random.default_rng(seed + i)
        x, _, meta, _, _ = sample_scene(
            rng,
            cfg=CFG,
            X=X,
            Y=Y,
            n_eddies=n_eddies,
            sampling_mode="full",
            single_only=False,
        )
        xs.append(np.concatenate([x, COORD_CHANNELS], axis=0))
        metas.append(meta)
    return torch.tensor(np.stack(xs)), metas

xb_multi, meta_multi = build_multiedy_regression_demo_batch(n_samples=4, n_eddies=3)

with torch.no_grad():
    pred_multi = reg_model(xb_multi.to(DEVICE)).cpu().numpy()

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for i, ax in enumerate(axes):
    ax.imshow(xb_multi[i, 0].numpy())

    for eddy in meta_multi[i]["eddies"]:
        ax.plot(km_to_pixel_x(eddy["x0_km"]), km_to_pixel_y(eddy["y0_km"]), marker="x", markersize=9, mew=2)

    pred_x_km, pred_y_km, pred_sigma_km = denormalize_regression_output(pred_multi[i])
    pred_center = (km_to_pixel_x(pred_x_km), km_to_pixel_y(pred_y_km))
    pred_circle = Circle(pred_center, pred_sigma_km / CFG.dx_km, fill=False, linestyle="--", linewidth=2)
    ax.add_patch(pred_circle)
    ax.plot(*pred_center, marker="+", markersize=12, mew=2)
    ax.set_title("Single-object regressor\napplied to 3-eddy scene")
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

## 9. Oceanographic interpretation

The synthetic experiment is simple, but the conclusions connect directly to satellite oceanography:

### 1. Small eddies are harder
As $\sigma$ decreases, the anomaly becomes sharper and occupies fewer pixels. The model sees less spatial support, so segmentation IoU and center/radius regression both degrade. This is a direct analog of **resolution limits** in satellite products.

### 2. Noise matters, but not all noise matters equally
Weak noise has modest impact. Strong noise reduces both segmentation and regression skill, especially when the noise scale overlaps the eddy scale. In real SSH, analogous contaminants include mapping error, internal tides, and unresolved background variability.

### 3. Sampling geometry matters
The difference between:
- **full/swath-like 2D coverage**, and
- **nadir-like 1D sampling**

is fundamental. A nadir track can miss the 2D geometry of an eddy even if it intersects the feature. That is why wide-swath SSH measurements are scientifically transformative for scale-dependent eddy identification.

### 4. Formulation matters
- **Segmentation** is naturally suited to multi-eddy scenes.
- **Single-object regression** is well posed for one eddy, but becomes ambiguous when multiple eddies are present.

### 5. What this notebook does *not* do
This is a controlled ML demonstration, **not** a full geophysical eddy detector. It omits:
- realistic frontogenesis and filaments
- internal tides and wave contamination
- anisotropy and advection
- dynamical constraints such as geostrophic balance
- true orbital sampling and revisit timing

Still, I hope this notebook is useful with a clean connection between ML/DL and ocean signals.

In [ ]:
print("Interpretation summary from this run")
print("-" * 60)
print(f"Size sweep: segmentation IoU from {size_seg_iou[0]:.3f} at sigma={size_values_km[0]} km "
      f"to {size_seg_iou[-1]:.3f} at sigma={size_values_km[-1]} km.")
print(f"Size sweep: regression center error from {size_reg_center[0]:.1f} km "
      f"to {size_reg_center[-1]:.1f} km across the same range.")

print(f"Noise sweep: segmentation IoU changed from {noise_seg_iou[0]:.3f} at noise=0 m "
      f"to {noise_seg_iou[-1]:.3f} at noise={noise_values_m[-1]:.2f} m.")
print(f"Sampling sweep: segmentation IoU full={sampling_seg_iou[0]:.3f}, "
      f"swath={sampling_seg_iou[1]:.3f}, nadir={sampling_seg_iou[2]:.3f}.")

print("\nTakeaway:")
print("Smaller eddies, noisier SSH, and sparser sampling all reduce recoverable eddy information.")

## 10. Exercises

Use the notebook as a starting point for your own OCNG 489/689 investigations.

### Exercise 1 — Change the $\sigma$ distribution
Bias the generator toward **smaller eddies** and retrain.  
Questions:
- Does segmentation IoU improve for small eddies?
- Does large-eddy performance degrade?
- How does this resemble a resolution-dependent prior in a training set?

**Hint:** modify `sigma_override` logic or replace the uniform draw in `sample_scene`.

---

### Exercise 2 — Add stronger correlated noise
Increase the correlated noise amplitude or length scale.  
Questions:
- Which breaks first: segmentation or regression?
- Is the model confusing smooth background variability with eddy structure?

**Hint:** increase `corr_noise_std` and/or `corr_length_range_km`.

---

### Exercise 3 — Train only on single eddies, test on multi-eddy scenes
This notebook already shows why direct regression becomes ill posed for multiple eddies.  
Extend that idea to segmentation:
- train segmentation only on single-eddy scenes
- test on 2–3 eddies
- inspect where merged or overlapping objects fail

**Hint:** set `single_only=True` in the segmentation dataset builder.

---

### Exercise 4 — Mimic nadir altimetry more aggressively
Retrain with only `sampling_mode="nadir"` or `sampling_mode="decimate"`.  
Questions:
- How much performance loss is due to **sampling sparsity** rather than model capacity?
- Can the model infer 2D eddy shape from essentially 1D information?

---

### Exercise 5 — Move toward more realistic eddies
Replace isotropic Gaussians with:
- elliptical Gaussians
- Gaussian rings
- eddies embedded in fronts
- time-evolving translating eddies

This is the natural bridge from a toy demo to a more realistic observing-system simulation experiment.

## References

- Chelton, D. B., Schlax, M. G., & Samelson, R. M. (2011). **Global observations of nonlinear mesoscale eddies**. *Progress in Oceanography*, 91(2), 167–216. https://doi.org/10.1016/j.pocean.2011.01.002  
- Fu, L.-L., & Ferrari, R. (2008). **Observing oceanic submesoscale processes from space**. *Eos, Transactions AGU*, 89(48). https://doi.org/10.1029/2008EO480003  
- NASA JPL SWOT Oceanography page: https://swot.jpl.nasa.gov/science/oceanography/  
- NASA PO.DAAC SWOT page: https://podaac.jpl.nasa.gov/SWOT  
- Zhang, D., Gade, M., Wang, W., & Zhou, H. (2023). **EddyDet: A Deep Framework for Oceanic Eddy Detection in Synthetic Aperture Radar Images**. *Remote Sensing*, 15(19), 4752. https://doi.org/10.3390/rs15194752  
- Martin, S. A., Manucharyan, G. E., & Klein, P. (2024). **Deep learning improves global satellite observations of ocean eddy dynamics**. *Geophysical Research Letters*, 51, e2024GL110059. https://doi.org/10.1029/2024GL110059  
- Archambault, T., Filoche, A., Charantonis, A., Bereziat, D., & Thiria, S. (2024). **Learning Sea Surface Height Interpolation From Multi-Variate Simulated Satellite Observations**. *Journal of Advances in Modeling Earth Systems*, 16, e2023MS004047. https://doi.org/10.1029/2023MS004047

### Note

The Gaussian generator in this notebook is an **idealized teaching model**, not a claim that real eddies are exactly Gaussian. Its value is that every generative parameter has a clear physical interpretation, making it useful for controlled ML experiments.